# Direct-CP fit with efficiency and background

A complete high-level CP workflow: define models, efficiency and background once, generate the pseudoexperiment with `generate_cp_toy`, and fit it with `CPFitSession`.


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    CPBackgroundSpec, CPFitSession, CPRealImag, CPToyBackground,
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    enable_x64, generate_cp_toy,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency
enable_x64()


In [ ]:
cp = CPRealImag(
    Parameter.coefficient("NR.x",1.0,bounds=(0.2,2.0),owner="NR",step=0.02),
    Parameter.coefficient("NR.y",0.2,bounds=(-1,1),owner="NR",step=0.02),
    Parameter.coefficient("NR.dx",0.08,bounds=(-0.4,0.4),owner="NR",step=0.01),
    Parameter.coefficient("NR.dy",-0.04,bounds=(-0.4,0.4),owner="NR",step=0.01),
)
plus_model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [Resonance("Kstar",(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),
     NonResonant(cp.for_charge(+1))],
    normalization_method="square-dalitz",normalization_resolution=200,normalization_pair=(0,2),
)
minus_model=DecayModel(
    DecayChannel("B-",("K-","pi-","pi+")),
    [Resonance("Kstar",(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),
     NonResonant(cp.for_charge(-1))],
    normalization_method="square-dalitz",normalization_resolution=200,normalization_pair=(0,2),
)
truth={p.name:p.value for p in plus_model.parameters}

eff=FunctionalEfficiency(lambda d:0.55+0.25*jnp.clip(d["s13"]/20.0,0,1))
bkg=FunctionalBackground(lambda d:0.4+0.6*jnp.clip(d["s23"]/25.0,0,1))


In [ ]:
plus_data,minus_data=generate_cp_toy(
    plus_model,minus_model,45_000,parameters=truth,
    plus_efficiency=eff,minus_efficiency=eff,
    signal_fraction=0.82,
    backgrounds=(CPToyBackground("comb",bkg),),
    seed=606,pool_size=280_000,
)
print("B+ / B-:",plus_data.size,minus_data.size)


In [ ]:
f_sig=Parameter("signal_fraction",0.74,bounds=(0.05,0.99),step=0.01)
session=CPFitSession(
    plus_model,minus_model,plus_data,minus_data,
    plus_efficiency=eff,minus_efficiency=eff,
    signal_fraction=f_sig,
    backgrounds=(CPBackgroundSpec("comb",bkg),),
)
start={p.name:p.value+0.05 for p in session.parameters if not p.fixed and p.name!="signal_fraction"}
start["signal_fraction"]=0.74
result=session.fit(start,simplex=True,ncall=55_000)
session.report(result,acceptance_weighted_fractions=True)
session.plot_projection(result,"s13")
plt.show()
